# Motion-stratified robustness figure

Robustness-tier check (CLAUDE.md, "Motion stratification") on claim 2: does
connectome similarity depend on head motion, and does the within-task >
between-task ordering survive when motion is held down? Reads
`output_data/motion_strata/*.tsv` (written by `run-motion-strata`) and plots
only — no similarity computation here, which lives in
`analysis/motion_strata.py`.

**Standalone figure, deliberately not placed in `connectome_figure.svg`** —
the domain panels' placement there was an explicit, user-requested exception
and does not generalize (CLAUDE.md).

No session in the gated population reaches `fd_mean > 0.3` (max 0.248), so
this tests *relative* motion differences inside an already low-motion cohort,
not high-motion data in general — the caption below carries that framing
verbatim into the figure folder, following the `network_quality_note.txt`
convention from `figure_connectomes.ipynb`.

Panels:
1. `motion_bins.png` — the six low/low, low/high, high/high x within/between-task
   bins, one group of bars per network, "cell" split (median split within each
   (subject, dataset) cell — orthogonal to both by construction).
2. `motion_balance.png` — pair-min usable duration and pair-min tSNR per bin,
   the audit that the motion split is not secretly a duration or a pure-tSNR
   confound.
3. `motion_permutation.png` — per-network permutation-test effect size
   (`median(low-low) - median(high-high)`), two-sided p-value, and per-subject
   replication count (of 6).


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_motion"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURE = invoke_config.get("analysis_measure", "pearson")

motion_dir = output_dir / "motion_strata"
motion_bins = pd.read_csv(motion_dir / "motion_strata.tsv", sep="\t")
motion_balance = pd.read_csv(motion_dir / "motion_balance.tsv", sep="\t")
motion_permutation = pd.read_csv(motion_dir / "motion_permutation.tsv", sep="\t")

print(f"📂 {PARCELLATION}, measure={MEASURE}: {len(motion_bins)} motion-bin rows, "
      f"{len(motion_permutation)} permutation rows")


📂 cneuromod2026, measure=pearson: 108 motion-bin rows, 9 permutation rows


In [2]:
def save_legend(handles, labels, name, default_size, ncol=None, fontsize=7):
    """Render `handles`/`labels` alone into `{name}` as a horizontal strip."""
    if not handles:
        return
    figsize = panel_size(f"figure_motion/{name}", default_size)
    fig = plt.figure(figsize=figsize, layout="constrained")
    fig.legend(
        handles, labels, loc="center",
        ncol=ncol or min(len(handles), 5), fontsize=fontsize, frameon=False,
    )
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / name} at {figsize} in")


In [3]:
# Shared color scheme (dataviz skill, "color-formula"): hue = task (categorical,
# 2 series, validated pair — blue/orange, worst adjacent CVD dE 24.7 light), alpha
# = motion pairing (ordinal: low-low -> low-high -> high-high increases apparent
# motion contrast, so darker/more opaque = more motion mismatch). This keeps every
# panel on the same two-hue system instead of six unrelated flat colors.
TASK_HUE = {"within-task": "#2a78d6", "between-task": "#eb6834"}
MOTION_ALPHA = {"low-low": 0.40, "low-high": 0.70, "high-high": 1.0}
MOTION_ORDER = ["low-low", "low-high", "high-high"]
TASK_ORDER = ["within-task", "between-task"]
BIN_ORDER = [f"{m}/{t}" for m in MOTION_ORDER for t in TASK_ORDER]


def bin_color(bin_label):
    motion, task = bin_label.split("/")
    return TASK_HUE[task], MOTION_ALPHA[motion]


In [4]:
# Panel 1 — motion_bins.png: six motion x task bins, "cell" split, all networks.
# Hue marks task (within/between), alpha marks motion pairing — so the panel
# itself shows the finding: within a hue, the three alpha steps barely move,
# because the motion effect is tiny next to the task effect.
figsize = panel_size("figure_motion/motion_bins.png", (6.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

cell_bins = motion_bins[motion_bins["split"] == "cell"]

x = np.arange(len(NETWORK_ORDER))
width = 0.13
legend_handles, legend_labels = [], []
if len(cell_bins):
    for i, bin_label in enumerate(BIN_ORDER):
        color, alpha = bin_color(bin_label)
        values = []
        for network in NETWORK_ORDER:
            row = cell_bins[(cell_bins["network"] == network) & (cell_bins["bin"] == bin_label)]
            values.append(row["median"].iloc[0] if len(row) else np.nan)
        bars = ax.bar(x + (i - 2.5) * width, values, width, color=color, alpha=alpha,
                       edgecolor="white", linewidth=0.4)
        legend_handles.append(bars[0])
        legend_labels.append(bin_label)
    ax.set_xticks(x)
    ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("median similarity (Fisher-z)")
    ax.spines[["top", "right"]].set_visible(False)
else:
    ax.text(0.5, 0.5, "no QC-covered sessions\n(smoke run, or too few cells)",
            ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(figure_dir / "motion_bins.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'motion_bins.png'} at {figsize} in")

save_legend(legend_handles, legend_labels, "motion_bins_legend.png", (6.0, 0.9), ncol=3)


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_bins.png at (6.0, 4.5) in


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_bins_legend.png at (6.0, 0.9) in


In [5]:
# Panel 2 — motion_balance.png: pair-min usable duration and pair-min tSNR per
# bin ("cell" split) — the audit that the motion split is not secretly a
# duration or tSNR confound (CLAUDE.md: fd_mean vs tsnr r=-0.68, so tSNR is
# expected to track the motion stratum; duration should not). Same hue/alpha
# scheme as motion_bins.png so a reader can carry the bin identity across panels.
figsize = panel_size("figure_motion/motion_balance.png", (6.0, 4.0))
fig, (ax_duration, ax_tsnr) = plt.subplots(1, 2, figsize=figsize, layout="constrained")

cell_balance = motion_balance[motion_balance["split"] == "cell"]
if len(cell_balance):
    ordered = cell_balance.set_index("bin").loc[BIN_ORDER].reset_index()
    positions = np.arange(len(BIN_ORDER))
    rgba_colors = [plt.matplotlib.colors.to_rgba(c, alpha=a)
                   for c, a in (bin_color(b) for b in BIN_ORDER)]

    ax_duration.bar(positions, ordered["median_min_duration_sec"], color=rgba_colors)
    ax_duration.set_xticks(positions)
    ax_duration.set_xticklabels(BIN_ORDER, rotation=90, fontsize=6)
    ax_duration.set_ylabel("median pair-min usable duration (s)")
    ax_duration.spines[["top", "right"]].set_visible(False)

    ax_tsnr.bar(positions, ordered["median_min_tsnr"], color=rgba_colors)
    ax_tsnr.set_xticks(positions)
    ax_tsnr.set_xticklabels(BIN_ORDER, rotation=90, fontsize=6)
    ax_tsnr.set_ylabel("median pair-min tSNR")
    ax_tsnr.spines[["top", "right"]].set_visible(False)
else:
    for sub_ax in (ax_duration, ax_tsnr):
        sub_ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=sub_ax.transAxes, color="0.5", fontsize=8)
        sub_ax.set_xticks([])
        sub_ax.set_yticks([])

fig.savefig(figure_dir / "motion_balance.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'motion_balance.png'} at {figsize} in")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_balance.png at (6.0, 4.0) in


In [6]:
# Panel 3 — motion_permutation.png: per-network permutation effect size
# (median(low-low) - median(high-high)) and per-subject replication count.
# This IS the headline number for the whole check, so it is designed to say
# what it found: a diverging blue/red encodes sign (positive = low motion more
# similar, as hypothesized; red = the opposite), desaturated because no network
# clears p<0.05 — full saturation is reserved for a real finding, not spent on
# noise. A summary line states the magnitude directly rather than making the
# reader infer "tiny" from bar length alone.
figsize = panel_size("figure_motion/motion_permutation.png", (5.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

POSITIVE_COLOR = "#8fb8e8"  # desaturated blue (diverging pair, muted: not significant)
NEGATIVE_COLOR = "#e8a3a2"  # desaturated red
SIGNIFICANT_COLOR = "#184f95"  # full-strength blue, reserved for p<0.05 (unused here)

if len(motion_permutation) and motion_permutation["observed_diff"].notna().any():
    ordered = motion_permutation.set_index("network").reindex(NETWORK_ORDER).reset_index()
    y = np.arange(len(ordered))

    def _bar_color(diff, p_value):
        if p_value is not None and p_value < 0.05:
            return SIGNIFICANT_COLOR
        return POSITIVE_COLOR if diff >= 0 else NEGATIVE_COLOR

    colors = [_bar_color(d, p) for d, p in zip(ordered["observed_diff"], ordered["p_value"])]
    ax.barh(y, ordered["observed_diff"], color=colors, edgecolor="white", linewidth=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(ordered["network"], fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0, color="0.35", linewidth=0.8)
    ax.set_xlabel("median(low-low) - median(high-high)\n(Fisher-z, ~ correlation units near 0)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.3f}"))

    # Anchored near the zero baseline (not the bar tip): the longest bars reach
    # close to the plot edge, and a tip-anchored label there would collide with
    # the y-axis tick labels ("SomMot", "subcortex") outside the axes.
    for i, row in ordered.iterrows():
        if pd.notna(row["n_subjects_replicating"]):
            align = "left" if row["observed_diff"] >= 0 else "right"
            ax.annotate(
                f"{int(row['n_subjects_replicating'])}/{int(row['n_subjects_total'])} subj",
                (0, i), fontsize=6, color="0.35",
                xytext=(6 if row["observed_diff"] >= 0 else -6, 0),
                textcoords="offset points", va="center", ha=align,
            )

    n_significant = int((ordered["p_value"] < 0.05).sum())
    max_abs_diff = float(ordered["observed_diff"].abs().max())
    n_positive = int((ordered["observed_diff"] > 0).sum())
    n_total = int(ordered["observed_diff"].notna().sum())
    n_perm = (
        int(ordered["n_permutations"].dropna().iloc[0])
        if ordered["n_permutations"].notna().any() else 0
    )
    coverage_note = (
        f"Motion effect is negligible and inconsistent in direction: max |{max_abs_diff:.3f}| "
        "across networks (well under 1% of the within-task/between-task gap of ~0.15-0.4), "
        f"{n_positive}/{n_total} networks positive, {n_significant}/{n_total} significant "
        f"(two-sided permutation test, N={n_perm}, p<0.05). "
        "Labels: subjects replicating the sign / subjects with both bins populated. "
        "This tests relative motion differences inside an already low-motion cohort "
        "(fd_mean max 0.248 mm over the gated population) — a null here is informative "
        "about these data, not evidence about high-motion data in general."
    )
else:
    ax.text(0.5, 0.5, "no permutation results\n(smoke run, or too few sessions)",
            ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    coverage_note = ""

fig.savefig(figure_dir / "motion_permutation.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'motion_permutation.png'} at {figsize} in")

note_path = figure_dir / "motion_note.txt"
note_path.write_text(coverage_note + "\n" if coverage_note else "")
print(f"✅ wrote {note_path}")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_permutation.png at (5.0, 4.5) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_note.txt
